# 09 · Учитель-промпт и несколько слоёв: цель без учителя-LoRA, которая переносится

**Зачем** (JOURNAL, 2026-10-31, прогон 08). Аффинная поправка в точке вызова с LoRA-учителем переносится
на новые типы требований (0.78 против 0.85 у LoRA), а без учителя — нет (0.61): узкое место — цель.
Здесь две цели без градиентного обучения и без LoRA:

| метод | цель | слои |
|---|---|---|
| `linmap_prompt` | Δh = h(та же модель с промптом explicit_state) − h(база), где учитель решил верно | один, по сетке |
| `linmap_stack` | цель `linmap` (центроид верных состояний базы) | все слои сразу |
| `linmap_prompt_stack` | цель `linmap_prompt` | все слои сразу |

Слои сразу — идея SimplexLoRA (2610.37206): бюджет ранга R = 8 × число слоёв и сила A распределены по слоям
на симплексе — **поровну** или **по важности** слоя (прирост лучшей одно-слойной конфигурации на val).

Данные — те же 110 промптов train; выбор на val; сетка силы расширена (1.5 была на краю). Проверка —
на **свежем** наборе переноса (seed 303): набор 202 уже повлиял на выбор этих методов. Результат **разведочный**.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="prompt-teacher-qwen3-1.7b", hypothesis="explore", notebook="notebooks/09_prompt_teacher.ipynb",
    notes="Учитель-промпт (explicit_state) и несколько слоёв с бюджетом по важности (SimplexLoRA) против LoRA.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "steering", "lora", "explicit_state", "linmap_lora", "linmap",
                         "linmap_prompt", "linmap_stack", "linmap_prompt_stack"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.ranks": [8, 16],
        "act.alphas": [1.0, 1.5, 2.0, 3.0],
        "episode.on_val": False,
        "transfer.seed": 303,
        "transfer.episode_methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_stack",
                                     "linmap_prompt_stack"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

### Подбор на val

In [ ]:
f = exp.state["fitted"]
t = f.get("linmap_prompt", {})
print(f"учитель {t.get('teacher')} на train: успех {t.get('teacher_train_success', float('nan')):.3f} "
      f"(база {t.get('base_train_success', float('nan')):.3f})")
for m in ("linmap_prompt", "linmap", "linmap_lora"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    display(Markdown(f"**{m}** — лучшие конфигурации на val"))
    cols = ["layer", "rank", "alpha", "train_r2", "success", "sensitivity", "specificity", "spec_drop", "fixed", "broken"]
    display(t.sort_values("success", ascending=False)[cols].head(8).round(3))

In [ ]:
for m in ("linmap_stack", "linmap_prompt_stack"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    imp = {L: round(g, 3) for L, g in f[m]["importance"].items()}
    display(Markdown(f"**{m}** — равномерно / по важности; важность слоя (прирост на val): {imp}"))
    cols = ["alloc", "alpha", "layers", "ranks", "weights", "success", "sensitivity", "specificity",
            "spec_drop", "fixed", "broken"]
    display(t[cols].round(3))

### Свежий набор переноса (seed 303): знакомые и новые типы требований

In [ ]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

### Цена: подготовка и вывод

In [ ]:
d = exp.backend.hidden_size
cost = []
lr = f["lora"]["row"]
cost.append({"method": "lora", "prep": "градиент по весам (копия fp32)", "prep_seconds": lr["train_seconds"],
             "params": lr["n_trainable"]})
for m in ("linmap_lora", "linmap", "linmap_prompt"):
    r, c = f[m]["row"], f[m].get("cost", {})
    if r:
        how = {"linmap_lora": "закрытая форма + учитель LoRA (+ цена LoRA)", "linmap": "закрытая форма",
               "linmap_prompt": "закрытая форма + учитель-промпт (генерация)"}[m]
        cost.append({"method": m, "prep": how, "prep_seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0),
                     "params": 2 * d * r["rank"] + d})
for m, parent in (("linmap_stack", "linmap"), ("linmap_prompt_stack", "linmap_prompt")):
    r = f[m]["row"]
    if r:
        c = f[parent].get("cost", {})
        cost.append({"method": m, "prep": f"как {parent}, {len(r['layers'])} слоёв ({r['alloc']})",
                     "prep_seconds": c.get("acts_seconds", 0) + c.get("fit_seconds", 0),
                     "params": sum(2 * d * k + d for k in r["ranks"])})
cost = pd.DataFrame(cost)
inf = exp.results_table("transfer:seen").set_index("method")[["tokens", "calls", "p50_ms"]]
cost.merge(inf, left_on="method", right_index=True, how="outer").reset_index(drop=True)

## Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))